# **EDA Notebook**



---
## 0. Setup Environment

In [2]:
# DO NOT MODIFY THE CODE IN THIS CELL
!pip install -q utstd

from utstd.folders import *
from utstd.ipyrenders import *

at = AtFolder(
    course_code=36106,
    assignment="AT3",
)
at.run()

import warnings
warnings.simplefilter(action='ignore')

ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
umap-learn 0.5.12 requires scikit-learn>=1.6, but you have scikit-learn 1.5.2 which is incompatible.
hdbscan 0.8.42 requires scikit-learn>=1.6, but you have scikit-learn 1.5.2 which is incompatible.
Mounted at /content/gdrive

You can now save your data files in: /content/gdrive/MyDrive/36106/assignment/AT3/data


---
## Student Information

In [3]:
# <Student to fill this section>
group_name = "Group 17"
student_name = "Chen-Tai Wang"
student_id = "25976996"

In [4]:
# Do not modify this code
print_tile(size="h1", key='group_name', value=group_name)

In [5]:
# Do not modify this code
print_tile(size="h1", key='student_name', value=student_name)

In [6]:
# Do not modify this code
print_tile(size="h1", key='student_id', value=student_id)

---
## 0. Python Packages

### 0.a Install Additional Packages

> If you are using additional packages, you need to install them here using the command: `! pip install <package_name>`

In [ ]:
# <Student to fill this section>

### 0.b Import Packages

In [7]:
import pandas as pd
import altair as alt
import numpy as np

---
## B. Data Understanding

In [8]:
# Do not modify this code
try:
  df = pd.read_csv(at.folder_path / "customer.csv")
except Exception as e:
  print(e)

### B.1 Explore Dataset

In [9]:
print(f"Shape: {df.shape}")
print(f"\nColumn types:\n{df.dtypes}")
print(f"\nFirst 5 rows:")
df.head()

Shape: (14275, 5)

Column types:
customer_id       object
person_id         object
store_id          object
territory_id      object
account_number    object
dtype: object

First 5 rows:


,customer_id,person_id,store_id,territory_id,account_number
0,4b5762b3-b7bf-4716-9bf9-5c1c9e1f375d,092707ff-4088-4416-ac86-14491f3120c2,NaN,0ad4c625-bb65-4376-8a41-0a65719b0db8,AW00019475
1,26bfba89-84b7-4b5b-9bab-caf7a2791006,fe043525-e0af-4578-a4b9-bfa7be28f4e4,NaN,25d73ad2-9e13-410b-8b0d-5f26e2b9e4f2,AW00022390
2,cf2cf4cc-fcce-4fe5-9400-78373420b525,ee25bbaf-5cdd-4ed4-92fe-7136c4e803a9,NaN,0ad4c625-bb65-4376-8a41-0a65719b0db8,AW00016320
3,83f558b8-bdab-4781-b112-55d2c275ff52,f0452f05-3690-4170-9fb0-fbf9995bbea7,NaN,2ac923e9-3043-4f5e-bae0-ad28089bf187,AW00026149
4,5edc53f4-7821-4a9b-9c27-a45dd2af713e,16c2532d-adb6-4a33-b0ea-71b87d83a904,NaN,0ad4c625-bb65-4376-8a41-0a65719b0db8,AW00021849


In [10]:
print(f"Missing values:\n{df.isnull().sum()}")
print(f"\nDuplicate rows: {df.duplicated().sum()}")
print(f"Unique customer_id: {df['customer_id'].nunique()}")
print(f"\nBasic stats:")
df.describe(include='all')

Missing values:
customer_id           0
person_id           237
store_id          13823
territory_id          0
account_number        0
dtype: int64

Duplicate rows: 3944
Unique customer_id: 10331

Basic stats:


,customer_id,person_id,store_id,territory_id,account_number
count,14275,14038,452,14275,14275
unique,10331,9232,160,4,9392
top,060c5593-b9c3-4c41-97bd-78040e780308,22b96ea8-51cf-4fce-b556-112654cfee06,af39eeb6-feb3-4df5-a2f1-407ecdc01998,2ac923e9-3043-4f5e-bae0-ad28089bf187,AW00019103
freq,3,4,6,5567,4


In [11]:
dataset_insights = """
The customer dataset contains 14,275 rows and 5 columns (customer_id, person_id, store_id, territory_id, account_number), all stored as string type.

Key findings:
- There are 3,944 exact duplicate rows, leaving 10,331 unique customers after deduplication.
- person_id has 237 missing values (1.7%), indicating some customers are not linked to individual persons.
- store_id has 13,823 missing values (96.8%), meaning the vast majority of customers are individual consumers (B2C) rather than store-based (B2B).
- Only 452 unique customers have a store_id, suggesting a small B2B segment.
- territory_id has no missing values — every customer is assigned to a sales territory.
- All IDs are UUID format strings, requiring joins with other tables for meaningful analysis.
"""

In [12]:
# Do not modify this code
print_tile(size="h3", key='dataset_insights', value=dataset_insights)

### B.2 Explore Feature of Interest — person_id

In [13]:
print(f"person_id coverage: {df['person_id'].notna().sum()} / {len(df)} ({df['person_id'].notna().mean():.1%})")
print(f"\nCustomers WITHOUT person_id (likely B2B/store): {df['person_id'].isna().sum()}")

# Relationship between person_id and store_id
print(f"\nHas person_id AND store_id: {(df['person_id'].notna() & df['store_id'].notna()).sum()}")
print(f"Has person_id but NO store_id: {(df['person_id'].notna() & df['store_id'].isna()).sum()}")
print(f"Has store_id but NO person_id: {(df['person_id'].isna() & df['store_id'].notna()).sum()}")
print(f"Has NEITHER: {(df['person_id'].isna() & df['store_id'].isna()).sum()}")

person_id coverage: 14038 / 14275 (98.3%)

Customers WITHOUT person_id (likely B2B/store): 237

Has person_id AND store_id: 215
Has person_id but NO store_id: 13823
Has store_id but NO person_id: 237
Has NEITHER: 0


In [15]:
feature_1_insights = """
person_id is present for 98.3% of customers (14,038 out of 14,275). It serves as the foreign key to the person table which contains demographic info such as email_promotion preference.

The relationship between person_id and store_id reveals two distinct customer segments:
- B2C individual customers: have person_id but no store_id (majority, ~ 96.8%)
- B2B store customers: have store_id, may or may not have person_id (~ 3.2%)

This segmentation (is_store_customer) could be a useful feature for classification, as B2B and B2C customers likely exhibit different repeat purchase patterns.
"""

In [16]:
# Do not modify this code
print_tile(size="h3", key='feature_1_insights', value=feature_1_insights)

### B.3 Explore Feature of Interest — store_id

In [18]:
print(f"store_id coverage: {df['store_id'].notna().sum()} / {len(df)} ({df['store_id'].notna().mean():.1%})")

# Deduplicate first to get accurate counts
df_dedup = df.drop_duplicates()
store_custs = df_dedup[df_dedup['store_id'].notna()]
print(f"\nUnique store customers (after dedup): {len(store_custs)}")
print(f"Unique store_ids: {store_custs['store_id'].nunique()}")

chart = alt.Chart(
    pd.DataFrame({'Customer Type': ['Individual (B2C)', 'Store (B2B)'],
                  'Count': [df_dedup['store_id'].isna().sum(), df_dedup['store_id'].notna().sum()]})
).mark_bar().encode(
    x='Customer Type',
    y='Count',
    color='Customer Type'
).properties(title='Customer Type Distribution', width=300, height=300)
chart

store_id coverage: 452 / 14275 (3.2%)

Unique store customers (after dedup): 334
Unique store_ids: 160


alt.Chart(...)

In [21]:
feature_2_insights = """
store_id is highly sparse — only 3.2% of customers (452 out of 14,275 unique) have a store_id, representing B2B/reseller accounts. The remaining 96.8% are individual consumers.

This extreme imbalance means store_id itself is not useful as a continuous feature, but the binary indicator (is_store_customer) captures meaningful segmentation.
B2B customers may have fundamentally different purchasing patterns — potentially higher order values but different repeat purchase behaviour compared to individual consumers.
"""

In [22]:
# Do not modify this code
print_tile(size="h3", key='feature_2_insights', value=feature_2_insights)

### B.4 Explore Feature of Interest — territory_id

In [23]:
df_dedup = df.drop_duplicates()
territory_dist = df_dedup['territory_id'].value_counts()
print(f"Territory distribution:\n{territory_dist}")
print(f"\nNumber of unique territories: {df_dedup['territory_id'].nunique()}")

chart = alt.Chart(
    territory_dist.reset_index().rename(columns={'territory_id': 'Territory', 'count': 'Count'})
).mark_bar().encode(
    x=alt.X('Territory', sort='-y'),
    y='Count'
).properties(title='Customer Distribution by Territory', width=400, height=300)
chart

Territory distribution:
territory_id
2ac923e9-3043-4f5e-bae0-ad28089bf187    4046
e0e3bac4-790e-4617-84b3-be0c2bdb7070    2173
0ad4c625-bb65-4376-8a41-0a65719b0db8    2074
25d73ad2-9e13-410b-8b0d-5f26e2b9e4f2    2038
Name: count, dtype: int64

Number of unique territories: 4


alt.Chart(...)

In [25]:
feature_n_insights = """
Customers are distributed across 4 territories (linked to France, Germany, Australia, and United Kingdom via the sales_territory table).
The distribution is relatively balanced, with no single territory dominating overwhelmingly.

Territory is a potentially important feature for repeat purchase prediction, as purchasing behaviour may vary by region due to cultural differences, market maturity,
and local competition. After joining with sales_territory, this can be encoded as a categorical feature.
"""

In [27]:
# Do not modify this code
print_tile(size="h3", key='feature_n_insights', value=feature_n_insights)